In [1]:
#importing libraries

import numpy as np #data wrangling
import pandas as pd #data wrangling
import seaborn as sns #static data visualization
import matplotlib.pyplot as plt #static data visualization
import yfinance as yf #public API - Yahoo Finance
from datetime import datetime, timedelta #time functions

import plotly.express as px #dynamic data visualization
import plotly.graph_objs as go #dynamic data visualization
from plotly.subplots import make_subplots
from dash import Dash, html, dash_table, dcc, callback, Output, Input, State #dashboard - data visualization

In [4]:
#importing dataset with top 5 companies by sectors of my interest and their tickers
df = pd.read_csv('C:/Users/luengoag/Downloads/finance_portfolio_dataset.csv', usecols=[0, 1])

In [5]:
#extracting openning, closing, volume information from Yahoo Finance API for x period of time with x days intervals for the dataframe
history_data = yf.download(df['index'][:5].tolist(), period='1y', interval='1d')

C:\Users\luengoag\AppData\Local\Temp\ipykernel_20460\42404345.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  history_data = yf.download(df['index'][:5].tolist(), period='1y', interval='1d')
[*********************100%***********************]  5 of 5 completed


In [6]:
#income = stock.financials
#balance = stock.balance_sheet
#cashflow = stock.cashflow

In [7]:
#columns of interest from yahoo finance API
columns_to_keep = [   
    'state',
    'country',
    'sector',
    'industryKey',
    'exchangeTimezoneName',
    'currency',
    'marketCap',
    'volume',
    'averageVolume',
    'averageDailyVolume3Month',
    'fiftyDayAverageChange',
    'fiftyDayAverageChangePercent',
    'twoHundredDayAverageChange',
    'twoHundredDayAverageChangePercent',
    'open',
    'close',
    'dayLow', 
    'dayHigh',
    'regularMarketPreviousClose',
    'regularMarketOpen',
    'regularMarketDayLow',
    'regularMarketDayHigh',
    'dividendRate',
    'dividendYield',
    'fiveYearAvgDividendYield',
    'profitMargins',
    'totalCash',
    'totalCashPerShare',
    'ebitda',
    'totalDebt',
    'quickRatio',
    'currentRatio',
    'totalRevenue',
    'debtToEquity',
    'revenuePerShare',
    'returnOnAssets',
    'returnOnEquity',
    'grossProfits', 
    'freeCashflow', 
    'operatingCashflow',
    'earningsGrowth',
    'revenueGrowth',
    'grossMargins',
    'ebitdaMargins',
    'operatingMargins',
    'averageAnalystRating',
    'recommendationMean',
    'recommendationKey'
]

In [8]:
#extracting all relevant information for each ticker from the dataframe
tickers = df['index']
info_list = []

for ticker in tickers:
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        row = {col: info.get(col, None) for col in columns_to_keep}
        row['symbol'] = ticker  # Add ticker column
        info_list.append(row)
    except Exception as e:
        print(f"Failed to get info for {ticker}: {e}")

info_df = pd.DataFrame(info_list)
info_df.set_index('symbol', inplace=True)

In [9]:
#separating in smaller datasets to use in the plots
closing_data = history_data['Close']
volume_data = history_data['Volume']

In [10]:
#lineplot for closing data for the tickers in the dataframe
closing_lineplot = px.line(closing_data,
              x=closing_data.index,
              y=closing_data.columns,
              title='1-Year Closing Price Comparison of Companies by category')

closing_lineplot.update_layout(paper_bgcolor='#343434', font_color='#FFF');

In [11]:
#lineplot for volume data for the tickers in the dataframe
vol_lineplot = px.line(volume_data,
              x=volume_data.index,
              y=volume_data.columns,
             title='1-Year Transations Volume Comparison of Companies by category')

vol_lineplot.update_layout(paper_bgcolor='#343434', font_color='#FFF');

In [22]:
#Input for companies that are in user's portfolio and the shares amount
portfolio = []

while True:
    ticker = input("Enter ticker (or press Enter to finish): ").strip().upper()
    
    # Stop when ticker is empty
    if ticker == "":
        break
    
    try:
        shares = float(input(f"Enter number of shares for {ticker}: "))
        portfolio.append({'Ticker': ticker, 'Shares': shares})
    except ValueError:
        print("Invalid number of shares. Please try again.")

# Convert list of dictionaries to DataFrame
full_portfolio = pd.DataFrame(portfolio)

print("\nYour Portfolio:")
print(df)

Enter ticker (or press Enter to finish):  V
Enter number of shares for V:  1
Enter ticker (or press Enter to finish):  



Your Portfolio:
       company_name index
0    JPMorgan Chase   JPM
1   Bank of America   BAC
2         Citigroup     C
3       Wells Fargo   WFC
4      U.S. Bancorp   USB
..              ...   ...
75    Goldman Sachs    GS
76   Morgan Stanley    MS
77   Charles Schwab  SCHW
78        BlackRock   BLK
79        CME Group   CME

[80 rows x 2 columns]


In [23]:
#Enriching the portfolio information with additional information
portfolio = pd.DataFrame({
    'Ticker': ['AAPL', 'AMZN', 'V', 'BRK-B', 'EXXT.DE'],
    'Shares': [0.99,10,2,1, 3.99]
})

info_columns = ['longName','sector','industryKey','regularMarketPrice',]
info_list = []

for _, row in portfolio.iterrows():
    ticker = row['Ticker']
    shares = row['Shares']
    
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        # Extract only the required fields
        data = {col: info.get(col, None) for col in info_columns}
        data['Ticker'] = ticker
        data['Shares'] = shares
        data['Value'] = data['regularMarketPrice'] * shares if data['regularMarketPrice'] is not None else None
        
        info_list.append(data)
    
    except Exception as e:
        print(f"Failed to get info for {ticker}: {e}")

# Create final DataFrame
enriched_portfolio = pd.DataFrame(info_list)
enriched_portfolio.set_index('Ticker', inplace=True)

enriched_portfolio.head()

,longName,sector,industryKey,regularMarketPrice,Shares,Value
Ticker,,,,,,
AAPL,Apple Inc.,Technology,consumer-electronics,201.00,0.99,198.9900
AMZN,"Amazon.com, Inc.",Consumer Cyclical,internet-retail,209.69,10.00,2096.9000
V,Visa Inc.,Financial Services,credit-services,338.57,2.00,677.1400
BRK-B,Berkshire Hathaway Inc.,Financial Services,insurance-diversified,484.85,1.00,484.8500
EXXT.DE,iShares NASDAQ-100 UCITS ETF (DE),None,None,183.28,3.99,731.2872


In [14]:
print("Total Portfolio Value:", enriched_portfolio['Value'].sum())

Total Portfolio Value: 4190.045


In [31]:
#Portfolio pie-plot element

#portfolio_pie = px.pie(
#    enriched_portfolio, #dataset
#    values='Value',
#    names=enriched_portfolio.index, 
#    title='Portfolio Allocation by Company',
#    hover_data=[enriched_portfolio['longName'],
#                enriched_portfolio['industryKey']],
#    hole=0.5
#)

#portfolio_pie.update_traces(textposition='outside', textinfo='percent+label')
#portfolio_pie.update_layout(paper_bgcolor='#343434', font_color='#FFF')

In [34]:
pie_trace = portfolio_pie.data[0]

def create_portfolio_figure(pie_trace, enriched_portfolio):
    
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "pie"}, {"type": "table"}]]
    )

    # Add pie chart
    fig.add_trace(pie_trace, row=1, col=1)

    # Create and add table
    fig.add_trace(
        go.Table(
            columnwidth=[150, 130, 100, 150, 200],
            header=dict(
                values=['Ticker','Company','Shares','Value','sector','Industry'],
                align='left',
                font=dict(size=12, color='white'),
                fill_color='darkslategray'
            ),
            cells=dict(
                values=[
                    enriched_portfolio.index,
                    enriched_portfolio['longName'],
                    enriched_portfolio['Shares'],
                    enriched_portfolio['Value'].round(2),
                    enriched_portfolio['sector'],
                    enriched_portfolio['industryKey']
                ],
                align='left',
                font=dict(size=11, color='black'),
                fill_color='lightgray',
                height=30
            )
        ),
        row=1, col=2
    )

    # Layout settings
    fig.update_layout(
        height=400,
#        width=1300,
        paper_bgcolor='#343434', 
        font_color='#FFF',
        title_text="Portfolio Allocation by Company",
        margin=dict(l=50, r=50, t=80, b=50)
    )

    return fig

#create_portfolio_figure(pie_trace, enriched_portfolio)

In [33]:
# App, assenbling all the objects
app = Dash()

app.layout = html.Div(
                      children=[
                        dcc.Graph(id='table', figure=create_portfolio_figure(pie_trace, enriched_portfolio)),
                        #dcc.Graph(id='portfolio_pie', figure=portfolio_pie),
                        dcc.Graph(id='close_lineplot', figure=closing_lineplot),
                        dcc.Graph(id='vol_lineplot', figure=vol_lineplot),
                    ])



# Run the server
if __name__ == '__main__':
    app.run(debug=True, port=8080)